<a href="https://colab.research.google.com/github/vtu30293-cmyk/ihlusecasetasks/blob/main/usecase3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# LLM + RAG + FINE-TUNING CHATBOT
# Google Colab - No API Key Required
# ============================================================

# Install libraries
!pip install -q transformers datasets sentence-transformers accelerate

# ============================================================
# IMPORTS
# ============================================================

from datasets import Dataset
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    pipeline
)
import numpy as np
import torch

print("Libraries loaded successfully!")


# ============================================================
# PART 1: TRAINING DATA FOR FINE-TUNING
# ============================================================

training_data = {
    "text": [
        "Question: What is Python?\nAnswer: Python is a programming language used for AI, machine learning, data science and automation.",

        "Question: What is Java?\nAnswer: Java is an object-oriented programming language commonly used for backend and enterprise applications.",

        "Question: What is Artificial Intelligence?\nAnswer: Artificial Intelligence enables computers to perform tasks that normally require human intelligence.",

        "Question: What is Machine Learning?\nAnswer: Machine Learning is a branch of AI that allows computers to learn patterns from data.",

        "Question: What is RAG?\nAnswer: RAG stands for Retrieval Augmented Generation. It retrieves relevant information from documents and provides it to an LLM.",

        "Question: What is Spring Boot?\nAnswer: Spring Boot is a Java framework used to build web applications and REST APIs.",

        "Question: What is MySQL?\nAnswer: MySQL is a relational database management system used to store structured data."
    ]
}

dataset = Dataset.from_dict(training_data)

print("\nTraining dataset created!")
print("Number of training examples:", len(dataset))


# ============================================================
# PART 2: LOAD SMALL LLM
# ============================================================

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT2 does not have a padding token
tokenizer.pad_token = tokenizer.eos_token

print("\nLLM loaded:", model_name)


# ============================================================
# PART 3: TOKENIZATION
# ============================================================

def tokenize_function(examples):

    tokens = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # Labels are required for language-model training
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("\nDataset tokenized successfully!")


# ============================================================
# PART 4: FINE-TUNING
# ============================================================

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",

    num_train_epochs=3,

    per_device_train_batch_size=1,

    learning_rate=5e-5,

    logging_steps=1,

    save_strategy="no",

    report_to="none"
)

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset
)

print("\nStarting fine-tuning...")

trainer.train()

print("\nFine-tuning completed!")


# ============================================================
# PART 5: RAG KNOWLEDGE BASE
# ============================================================

documents = [

    "Python is a programming language used for artificial intelligence, machine learning, data science, web development and automation.",

    "Java is an object-oriented programming language used for backend development, enterprise applications and Android development.",

    "Artificial Intelligence is a technology that enables computers to perform tasks that normally require human intelligence.",

    "Machine Learning is a branch of Artificial Intelligence that allows computers to learn patterns from data.",

    "RAG stands for Retrieval Augmented Generation. RAG retrieves relevant information from documents and provides that information to an LLM before generating an answer.",

    "Spring Boot is a Java framework used to develop web applications and REST APIs.",

    "MySQL is a relational database management system used to store and manage structured data.",

    "HTML is a markup language used to create the structure of web pages.",

    "CSS is used to style and design web pages.",

    "JavaScript is a programming language used to make web pages interactive."
]


# ============================================================
# PART 6: CREATE RAG EMBEDDINGS
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

print("\nRAG embeddings created!")


# ============================================================
# PART 7: CREATE FINE-TUNED LLM PIPELINE
# ============================================================

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("\nFine-tuned LLM ready!")


# ============================================================
# PART 8: RAG + FINE-TUNED LLM
# ============================================================

def chatbot(question):

    # ----------------------------------------
    # RAG: Convert question into embedding
    # ----------------------------------------

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )[0]


    # ----------------------------------------
    # RAG: Find similarity
    # ----------------------------------------

    similarities = np.dot(
        document_embeddings,
        question_embedding
    )


    # ----------------------------------------
    # RAG: Find best document
    # ----------------------------------------

    best_index = np.argmax(similarities)

    context = documents[best_index]


    # ----------------------------------------
    # Send retrieved context to fine-tuned LLM
    # ----------------------------------------

    prompt = f"""
You are a helpful AI assistant.

Use the information below to answer the question.

Information:
{context}

Question:
{question}

Answer:
"""


    # ----------------------------------------
    # Generate answer
    # ----------------------------------------

    result = generator(
        prompt,

        max_new_tokens=60,

        do_sample=True,

        temperature=0.7,

        pad_token_id=tokenizer.eos_token_id
    )


    # ----------------------------------------
    # Extract answer
    # ----------------------------------------

    full_text = result[0]["generated_text"]

    answer = full_text[len(prompt):].strip()

    return answer, context


# ============================================================
# PART 9: CHATBOT
# ============================================================

print("\n")
print("==============================================")
print("       🤖 LLM + RAG + FINE-TUNING")
print("==============================================")
print("Chatbot is ready!")
print("Ask a question.")
print("Type 'bye' to exit.")
print("==============================================\n")


while True:

    question = input("You: ")

    if question.lower() == "bye":

        print("Bot: Goodbye! 👋")

        break


    answer, context = chatbot(question)


    print("\nBot:", answer)

    print("\nRetrieved Context:")
    print(context)

    print("\n" + "-" * 50 + "\n")

Libraries loaded successfully!

Training dataset created!
Number of training examples: 7


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...


Bot: What is Python?

Retrieved Context:
Python is a programming language used for artificial intelligence, machine learning, data science, web development and automation.

--------------------------------------------------



[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: Java is an object-oriented programming language used for enterprise development, enterprise applications and Android development.

Retrieved Context:
Java is an object-oriented programming language used for backend development, enterprise applications and Android development.

--------------------------------------------------



[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: 

Retrieved Context:
RAG stands for Retrieval Augmented Generation. RAG retrieves relevant information from documents and provides that information to an LLM before generating an answer.

--------------------------------------------------

